In [4]:
pip uninstall -y speechbrain parselmouth googleads


Found existing installation: speechbrain 1.0.3Note: you may need to restart the kernel to use updated packages.



Uninstalling speechbrain-1.0.3:
  Successfully uninstalled speechbrain-1.0.3


In [5]:

!pip install torch torchaudio numpy scipy librosa


Defaulting to user installation because normal site-packages is not writeable


In [6]:
!pip install speechbrain --no-deps

Defaulting to user installation because normal site-packages is not writeable


In [1]:
# features.py
import numpy as np
import librosa
import parselmouth

SR = 16000

def extract_behavioral_features(file_path, sr=SR, n_mfcc=20):
    # load, trim, normalize
    y, sr = librosa.load(file_path, sr=sr, mono=True)
    y, _ = librosa.effects.trim(y, top_db=20)
    if np.max(np.abs(y)) > 0:
        y = y / np.max(np.abs(y))

    # MFCC mean + delta/delta-delta
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    mfcc_mean = np.mean(mfcc, axis=1)            # n_mfcc
    mfcc_delta = np.mean(librosa.feature.delta(mfcc), axis=1)
    mfcc_deltadelta = np.mean(librosa.feature.delta(mfcc, order=2), axis=1)

    # Parselmouth for pitch, jitter, shimmer
    snd = parselmouth.Sound(file_path)
    pitch = snd.to_pitch()
    pitch_values = pitch.selected_array['frequency']
    pitch_values = pitch_values[pitch_values > 0]
    pitch_mean = float(np.mean(pitch_values)) if len(pitch_values) > 0 else 0.0
    pitch_std = float(np.std(pitch_values)) if len(pitch_values) > 0 else 0.0

    point_process = parselmouth.praat.call(snd, "To PointProcess (periodic, cc)", 75, 500)
    try:
        jitter_local = float(parselmouth.praat.call(point_process, "Get jitter (local)", 0, 0, 0.0001, 0.02, 1.3))
    except Exception:
        jitter_local = 0.0
    try:
        shimmer_local = float(parselmouth.praat.call([snd, point_process], "Get shimmer (local)", 0, 0, 0.0001, 0.02, 1.3, 1.6))
    except Exception:
        shimmer_local = 0.0

    # energy / RMS
    energy = float(np.mean(y**2))
    rms = float(np.mean(librosa.feature.rms(y=y)))

    # speaking rate approx (onset count / duration)
    onset_env = librosa.onset.onset_strength(y=y, sr=sr)
    onsets = librosa.onset.onset_detect(onset_envelope=onset_env, sr=sr)
    duration = librosa.get_duration(y=y, sr=sr)
    speaking_rate = float(len(onsets) / duration) if duration > 0 else 0.0

    # pause ratio: fraction of frames below energy threshold
    frames = librosa.util.frame(y, frame_length=1024, hop_length=512)
    frame_rms = np.sqrt(np.mean(frames**2, axis=0) + 1e-8)
    pause_ratio = float(np.mean(frame_rms < (0.2 * np.mean(frame_rms))))

    feat = np.concatenate([
        mfcc_mean, mfcc_delta, mfcc_deltadelta,
        [pitch_mean, pitch_std, jitter_local, shimmer_local, energy, rms, speaking_rate, pause_ratio]
    ])
    return feat  #  (n_mfcc*3 + 12) dimension


In [6]:
import os
os.makedirs("pretrained_models/speechbrain_ecapa_voxceleb", exist_ok=True)


In [7]:
import os
import torch
from speechbrain.pretrained import EncoderClassifier
from huggingface_hub import snapshot_download
import shutil

# 1️⃣ Download the model (no symlinks)
local_dir = snapshot_download(
    repo_id="speechbrain/spkrec-ecapa-voxceleb",
    local_dir="pretrained_models/speechbrain_ecapa_voxceleb",
    local_dir_use_symlinks=False
)

# 2️⃣ Ensure directory exists
savedir = "pretrained_models/speechbrain_ecapa_voxceleb"
os.makedirs(savedir, exist_ok=True)

# 3️⃣ Manually copy 'custom.py' if missing
src_custom = os.path.join(local_dir, "custom.py")
dst_custom = os.path.join(savedir, "custom.py")

if os.path.exists(src_custom) and not os.path.exists(dst_custom):
    shutil.copy(src_custom, dst_custom)
    print("✅ Copied custom.py manually.")
else:
    print("ℹ️ custom.py already exists or not needed.")

# 4️⃣ Load the model safely
spk_model = EncoderClassifier.from_hparams(
    source=savedir,
    savedir=savedir,
    run_opts={"device": "cuda" if torch.cuda.is_available() else "cpu"}
)

print("✅ ECAPA model loaded successfully (Windows-safe, no symlinks).")


Fetching 10 files: 100%|██████████| 10/10 [00:00<00:00, 4887.90it/s]


ℹ️ custom.py already exists or not needed.
✅ ECAPA model loaded successfully (Windows-safe, no symlinks).


In [24]:
import os
import torch
import torchaudio
import numpy as np
import librosa
from speechbrain.pretrained import EncoderClassifier
from huggingface_hub import snapshot_download

# -----------------------------------------------------------
# 1️⃣ Prepare environment & model
# -----------------------------------------------------------
os.environ["HF_HUB_DISABLE_SYMLINKS"] = "1"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

savedir = "pretrained_models/speechbrain_ecapa_voxceleb"
if not os.path.exists(savedir):
    print("📦 Downloading ECAPA-TDNN model...")
    snapshot_download(
        repo_id="speechbrain/spkrec-ecapa-voxceleb",
        local_dir=savedir,
        local_dir_use_symlinks=False
    )

spk_model = EncoderClassifier.from_hparams(
    source=savedir,
    savedir=savedir,
    run_opts={"device": "cuda" if torch.cuda.is_available() else "cpu"}
)
print("✅ ECAPA-TDNN model loaded successfully.")

# -----------------------------------------------------------
# 2️⃣ Audio preprocessing function
# -----------------------------------------------------------
def preprocess_audio(file_path, target_sr=16000):
    """Load, trim silence, normalize, and resample."""
    y, sr = librosa.load(file_path, sr=target_sr)
    y, _ = librosa.effects.trim(y, top_db=30)  # remove silence
    if len(y) < 16000:
        y = np.pad(y, (0, 16000 - len(y)))  # pad short clips
    y = y / (np.max(np.abs(y)) + 1e-8)
    return torch.tensor(y).unsqueeze(0)  # [1, time]

# -----------------------------------------------------------
# 3️⃣ Extract SpeechBrain embeddings
# -----------------------------------------------------------
def get_embedding(file_path):
    signal = preprocess_audio(file_path)
    emb = spk_model.encode_batch(signal)
    emb = emb.squeeze().detach().cpu().numpy().flatten()  # Ensure shape (192,)
    emb = emb / (np.linalg.norm(emb) + 1e-10)
    return emb

def verify_speaker(file1, file2, threshold=0.45):
    emb1 = get_embedding(file1)
    emb2 = get_embedding(file2)
    score = np.dot(emb1, emb2) / (np.linalg.norm(emb1) * np.linalg.norm(emb2))
    print(f"🔍 Similarity Score: {score:.3f}")
    if score > threshold:
        print("✅ Same speaker (match confirmed!)")
    else:
        print("❌ Different speaker")
    return score


# -----------------------------------------------------------
# 5️⃣ Example usage
# -----------------------------------------------------------
file1 = "User_Voice/U002/02.wav"
file2 = "User_Voice/U002/07.wav"

verify_speaker(file1, file2, threshold=0.40)


✅ ECAPA-TDNN model loaded successfully.
🔍 Similarity Score: 0.213
❌ Different speaker


0.21274607

In [25]:
import os, numpy as np, torch, librosa

device = "cuda" if torch.cuda.is_available() else "cpu"

def get_embedding_file(file_path, use_encode_file=True):
    """
    Robust embedding extraction:
      - loads & trims audio (librosa)
      - ensures mono 16k
      - calls spk_model.encode_file OR encode_batch
      - returns a 1D numpy vector (L2-normalized)
    """
    # preprocess: load and trim silence (keeps float32)
    y, sr = librosa.load(file_path, sr=16000, mono=True)
    y, _ = librosa.effects.trim(y, top_db=30)
    if len(y) < 1600:
        # avoid extremely short files
        y = np.pad(y, (0, max(0, 1600 - len(y))))
    # normalize
    y = y / (np.max(np.abs(y)) + 1e-9)
    
    # Try encode_file (speechbrain convenient API) else use encode_batch
    try:
        if use_encode_file:
            emb_t = spk_model.encode_file(file_path)  # may accept str
            emb = emb_t.squeeze().cpu().numpy()
        else:
            tensor = torch.tensor(y, dtype=torch.float32).unsqueeze(0)  # [1, time]
            emb_t = spk_model.encode_batch(tensor.to(device))
            emb = emb_t.squeeze().detach().cpu().numpy().flatten()
    except Exception as e:
        # fallback to batch with waveform if encode_file fails
        tensor = torch.tensor(y, dtype=torch.float32).unsqueeze(0)
        emb_t = spk_model.encode_batch(tensor.to(device))
        emb = emb_t.squeeze().detach().cpu().numpy().flatten()
    # L2-normalize
    emb = emb / (np.linalg.norm(emb) + 1e-10)
    return emb


In [26]:
import os
from collections import defaultdict

DATA_ROOT = "User_Voice"   # your folder with U001..U023
users = sorted([d for d in os.listdir(DATA_ROOT) if os.path.isdir(os.path.join(DATA_ROOT,d))])

# collect embeddings per user
embs_by_user = defaultdict(list)
all_files = []

for u in users:
    folder = os.path.join(DATA_ROOT, u)
    wavs = sorted([f for f in os.listdir(folder) if f.lower().endswith(".wav")])
    for w in wavs:
        path = os.path.join(folder, w)
        try:
            emb = get_embedding_file(path)
            embs_by_user[u].append(emb)
            all_files.append((u, path))
        except Exception as e:
            print("ERR embedding", path, e)

# build per-user centroid (average embedding)
user_centroid = {}
for u, L in embs_by_user.items():
    user_centroid[u] = np.mean(np.stack(L, axis=0), axis=0)
    user_centroid[u] /= (np.linalg.norm(user_centroid[u]) + 1e-10)

print("Computed embeddings and centroids for", len(user_centroid), "users.")


Computed embeddings and centroids for 23 users.


In [27]:
from itertools import combinations
intra_sims = []   # similarity of each file to its user's centroid
inter_sims = []   # similarity of each file to other users' centroids (impostor)

for u, L in embs_by_user.items():
    centroid = user_centroid[u]
    for emb in L:
        sim_g = float(np.dot(emb, centroid))
        intra_sims.append(sim_g)
        # compare to *other* centroids (a few or all)
        for v, c_v in user_centroid.items():
            if v == u: 
                continue
            inter_sims.append(float(np.dot(emb, c_v)))

# summary stats
def stats(arr):
    return np.min(arr), np.percentile(arr,5), np.percentile(arr,25), np.median(arr), np.mean(arr), np.percentile(arr,75), np.percentile(arr,95), np.max(arr)

print("INTRA (genuine) sims stats (min,5%,25%,median,mean,75%,95%,max):")
print(stats(intra_sims))
print("INTER (impostor) sims stats:")
print(stats(inter_sims))


INTRA (genuine) sims stats (min,5%,25%,median,mean,75%,95%,max):
(0.32460129261016846, 0.5244163304567337, 0.6771266758441925, 0.7602522969245911, 0.7303804532341335, 0.8009676486253738, 0.8641576677560806, 0.8792554140090942)
INTER (impostor) sims stats:
(-0.15832717716693878, 0.00797699303366244, 0.08976202644407749, 0.1557312235236168, 0.17514401152776002, 0.24863815307617188, 0.3960552543401718, 0.5994290709495544)


In [28]:
def identify_by_centroid(test_file):
    emb = get_embedding_file(test_file)
    best_user, best_sim = None, -1.0
    for u, c in user_centroid.items():
        s = float(np.dot(emb, c))
        if s > best_sim:
            best_sim = s
            best_user = u
    return best_user, best_sim

# Example: verify one file
test = "User_Voice/U002/02.wav"
user, sim = identify_by_centroid(test)
print("identified:", user, "sim:", sim)


identified: U002 sim: 0.5946459770202637


In [29]:
a = get_embedding_file("User_Voice/U002/02.wav")
b = get_embedding_file("User_Voice/U002/07.wav")
# compare to centroid
centroid_u2 = user_centroid["U002"]
print("sim a<->b:", float(np.dot(a,b)))
print("sim a<->centroid:", float(np.dot(a, centroid_u2)))
print("sim b<->centroid:", float(np.dot(b, centroid_u2)))
# compare to top-3 centroids
sims = [(u, float(np.dot(a, c))) for u,c in user_centroid.items()]
sims_sorted = sorted(sims, key=lambda x: -x[1])[:5]
print("top 5 sims for file a:", sims_sorted)


sim a<->b: 0.21274606883525848
sim a<->centroid: 0.5946459770202637
sim b<->centroid: 0.5424617528915405
top 5 sims for file a: [('U002', 0.5946459770202637), ('U017', 0.4960174858570099), ('U005', 0.3931563198566437), ('U019', 0.34773701429367065), ('U008', 0.3359297215938568)]


In [30]:
def verify_speaker(test_file, claimed_user=None, threshold=0.50):
    emb = get_embedding_file(test_file)
    
    if claimed_user:
        sim = float(np.dot(emb, user_centroid[claimed_user]))
        result = "✅ SAME speaker" if sim > threshold else "❌ Different speaker"
        print(f"[{claimed_user}] Sim={sim:.3f} → {result}")
        return sim, result
    else:
        # Identification mode — find the closest match
        sims = {u: float(np.dot(emb, c)) for u, c in user_centroid.items()}
        best_user = max(sims, key=sims.get)
        best_sim = sims[best_user]
        print(f"🔍 Identified: {best_user} (Sim={best_sim:.3f})")
        result = "✅ SAME speaker" if best_sim > threshold else "❌ Unknown speaker"
        print(result)
        return best_user, best_sim


In [62]:
verify_speaker("User_Voice/U007/02.wav", claimed_user="U007", threshold=0.50)


[U007] Sim=0.778 → ✅ SAME speaker


(0.7781375050544739, '✅ SAME speaker')

In [63]:
!pip install sounddevice


In [64]:
import sounddevice as sd
from scipy.io.wavfile import write

def record_voice(output_path="live_test.wav", duration=4, sr=16000):
    """Record audio from the default microphone."""
    print(f"🎙️ Recording {duration} seconds… Speak now!")
    audio = sd.rec(int(duration * sr), samplerate=sr, channels=1, dtype='float32')
    sd.wait()
    write(output_path, sr, audio)
    print(f"✅ Saved recording to {output_path}")
    return output_path

def verify_speaker(test_file, claimed_user=None, threshold=0.50):
    emb = get_embedding_file(test_file)
    
    if claimed_user:
        sim = float(np.dot(emb, user_centroid[claimed_user]))
        result = "✅ SAME speaker" if sim > threshold else "❌ Different speaker"
        print(f"[{claimed_user}] Sim={sim:.3f} → {result}")
        return sim, result
    else:
        # Identification mode — find the closest match
        sims = {u: float(np.dot(emb, c)) for u, c in user_centroid.items()}
        best_user = max(sims, key=sims.get)
        best_sim = sims[best_user]
        print(f"🔍 Identified: {best_user} (Sim={best_sim:.3f})")
        result = "✅ SAME speaker" if best_sim > threshold else "❌ Unknown speaker"
        print(result)
        return best_user, best_sim


In [71]:
# 1️⃣ Record a short sample
live_path = record_voice("live_test.wav", duration=4)

# 2️⃣ Identify or verify
verify_speaker(live_path, claimed_user="U005", threshold=0.50)


🎙️ Recording 4 seconds… Speak now!
✅ Saved recording to live_test.wav
[U005] Sim=0.531 → ✅ SAME speaker


(0.5307281017303467, '✅ SAME speaker')